In [2]:
from yolo.src.yolo.yolo_onnx import YOLOParams, YOLOPredictor
from yolo.src.yolo.yolo_onnx.backends.onnxruntime import ONNXRuntimeBackend, ONNXRuntimeParams
import os
from dotenv import load_dotenv
from pathlib import Path
from src.visual_security.analyzer import YOLOAnalyzer, FoundryGPT4oAnalyzer, AzureAIVisionAnalyzer, SafetyAnalyzerPipeline

In [ ]:
load_dotenv()

TEST_IMAGE = "C:/Users/a.damicis/Documents/projects/visual_security/data/test/images/10_JPG_jpg.rf.2e98bae6286cbb711d31db9c3939849a.jpg"
YOLO_WEIGHTS = "runs/detect/train/weights/best.onnx"

analyzers = []

# 1. YOLO (Locale)
if Path(YOLO_WEIGHTS).exists():
    analyzers.append(YOLOAnalyzer(model_path=YOLO_WEIGHTS))
else:
    print("⚠️ YOLO ONNX non trovato, salto il test locale.")

# Aggiungiamo Foundry GPT-4o
if os.getenv("AZURE_OPENAI_KEY"):
    analyzers.append(FoundryGPT4oAnalyzer(api_key=os.getenv("AZURE_OPENAI_KEY"), endpoint=os.getenv("AZURE_OPENAI_URL")))

# 3. Azure AI Vision (Deterministico)
if os.getenv("AZURE_VISION_KEY"):
    analyzers.append(
        AzureAIVisionAnalyzer(api_key=os.getenv("AZURE_VISION_KEY"), endpoint=os.getenv("AZURE_VISION_ENDPOINT"))
    )

pipeline = SafetyAnalyzerPipeline(analyzers)
if Path(TEST_IMAGE).exists():
    print(f"🚀 Avvio analisi su {TEST_IMAGE}...")
    results = pipeline.run(TEST_IMAGE)
    pipeline.print_report(results)
else:
    print(f"⚠️ Immagine non trovata in {TEST_IMAGE}. Carica un'immagine per testare.")